# 20 — Kiểm chứng triển khai trên MNIST và CIFAR-10

CB/LRE/MWN được kiểm tra công thức và chạy nhỏ trên dữ liệu gốc. Đây là kiểm chứng quy mô nhỏ với SmallCNN, không tái lập đầy đủ điểm số paper.

Bản sửa thêm thử lại khi Windows tạm khóa `epochs.csv`. Các fit đã hoàn tất và hợp lệ được dùng lại; fit bị ngắt chạy lại từ đầu fit đó. Notebook thực thi có output và mã sửa được lưu cùng proof.

**Cách chạy:** thay file notebook20 cũ bằng file này trong `BAF_project/notebooks`, đóng CSV đang mở bằng Excel/Preview, chọn kernel ml-gpu rồi Run All. Dừng notebook20/22 khác trước khi chạy.

Cấu hình mặc định giữ nguyên. `RESUME_RUN = None` chọn run bằng identity của cấu hình, source và môi trường hiện tại; không ưu tiên run từ log. Chỉ điền `RESUME_RUN` khi muốn resume một run cũ có manifest khớp hoàn toàn và nằm đúng đường dẫn suy ra từ identity. Thư mục run đã di chuyển bị từ chối trước khi launch vì reporting.launch dùng đường dẫn canonical. Không cần notebook22.

Output hiện có là bằng chứng của lần chạy trước bản sửa chọn run này. Luồng chọn run mới được kiểm tra nhẹ, không chạy lại huấn luyện.


In [4]:
from pathlib import Path
import hashlib
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "supplement" / "reporting.py").is_file():
    raise RuntimeError("Dat notebook trong BAF_project/notebooks va mo tu project do.")

sys.path.insert(0, str(ROOT / "supplement"))
import reporting

reporting.setup(ROOT)
print("Python:", sys.executable)
print("Project:", ROOT)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project


In [5]:
CONFIG = {
    "datasets": ["mnist", "cifar10"],
    "majority": 1000,
    "imbalance": 100,
    "epochs": 10,
    "seeds": [42, 317, 2026],
    "threads": 4,
    "device": "auto",
}

# None: derive the run from current config/source/environment only.
# Set an explicit existing canonical run path to request resume; full identity must match.
RESUME_RUN = None

source_paths = [
    ROOT / "src/baf.py",
    ROOT / "src/gru.py",
    ROOT / "supplement/reporting.py",
    ROOT / "supplement/training.py",
]
IDENTITY = {
    "version": reporting.VERSION,
    "tag": "original_checks",
    "config": CONFIG,
    "source": {
        str(path.relative_to(ROOT)): reporting.digest(path)
        for path in source_paths
    },
    "environment": reporting.environment(),
}
def select_original_check_run(root, identity, resume_run=None):
    """Select by the complete current identity without writing run evidence."""
    signature = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode()
    ).hexdigest()[:12]
    expected = (
        Path(root).resolve() / "results/advisor_runs" / ("original_checks_" + signature)
    ).resolve()
    selected = Path(resume_run).resolve() if resume_run is not None else expected
    manifest = selected / "manifest.json"
    if resume_run is not None and not manifest.is_file():
        raise FileNotFoundError(f"Explicit resume requires an existing manifest: {manifest}")
    if manifest.is_file():
        previous = reporting.read_json(manifest)
        if json.dumps(previous, sort_keys=True) != json.dumps(identity, sort_keys=True):
            changed = sorted(
                key for key in set(previous) | set(identity)
                if json.dumps(previous.get(key), sort_keys=True)
                != json.dumps(identity.get(key), sort_keys=True)
                or (key in previous) != (key in identity)
            )
            raise RuntimeError("Run identity differs: " + ", ".join(changed))
    elif selected.exists() and any(selected.iterdir()):
        raise RuntimeError(f"Nonempty run directory has no identity manifest: {selected}")
    # reporting.launch uses the same deterministic location. Reject a relocated
    # directory before launching anything; never launch a different run silently.
    if selected != expected:
        raise RuntimeError(
            "Resume path is not the identity-derived location used by reporting.launch: "
            + str(expected)
        )
    return selected


RUN_DIR = select_original_check_run(ROOT, IDENTITY, RESUME_RUN)

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run:", RUN_DIR)
print("Fit da hoan tat:", len(list(RUN_DIR.glob("*/*/COMPLETE.json"))))
print("Config:", CONFIG)


Run: D:\UIT\DS317\BAF_project\results\advisor_runs\original_checks_582e4a8ada1a
Fit da hoan tat: 10
Config: {'datasets': ['mnist', 'cifar10'], 'majority': 1000, 'imbalance': 100, 'epochs': 10, 'seeds': [42, 317, 2026], 'threads': 4, 'device': 'auto'}


In [6]:
worker_setup = """
import os
import sys
from pathlib import Path

ROOT = Path(os.environ["BAF_PROJECT_ROOT"])
RUN_DIR = Path(os.environ["BAF_RUN_DIR"])
sys.path.insert(0, str(ROOT / "supplement"))

import reporting
import training

reporting.setup(ROOT)
CONFIG = reporting.read_json(RUN_DIR / "manifest.json")["config"]
print("RESUME_ORIGINAL_RUN", RUN_DIR)
"""

worker_patch = """
from pathlib import Path
import os
import tempfile
import time


def write_csv_with_retry(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = frame.to_csv(index=False).encode("utf-8")

    handle, name = tempfile.mkstemp(
        prefix=path.name + ".",
        suffix=".tmp",
        dir=path.parent,
    )
    temporary = Path(name)
    try:
        with os.fdopen(handle, "wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())

        for attempt in range(51):
            try:
                temporary.replace(path)
                return
            except PermissionError as error:
                if attempt == 50:
                    raise PermissionError(
                        f"Windows van khoa {path}. Dong file CSV trong Excel, "
                        "VS Code/Preview, roi chay lai notebook nay. "
                        "Cac fit COMPLETE van duoc giu nguyen."
                    ) from error
                time.sleep(0.1)
    finally:
        try:
            temporary.unlink(missing_ok=True)
        except PermissionError:
            pass


reporting.write_csv = write_csv_with_retry
print("CSV_RETRY_PATCH_ACTIVE: toi da 5 giay, giu cach ghi atomic")
"""

worker_run = """
table = training.run_original(ROOT, RUN_DIR, CONFIG)
print(table.to_string(index=False))
print("ORIGINAL_DATASET_CHECK_READY", RUN_DIR)
"""

# Luu patch nhu evidence rieng, khong sua source/manifest/fit cu.
patch_hash = hashlib.sha256(worker_patch.encode()).hexdigest()
patch_file = RUN_DIR / ("csv_retry_patch_" + patch_hash[:12] + ".py")
if patch_file.exists():
    if patch_file.read_text(encoding="utf-8") != worker_patch:
        raise RuntimeError("Patch evidence da thay doi.")
else:
    patch_file.write_text(worker_patch, encoding="utf-8")

reporting.save_json(
    RUN_DIR / ("csv_retry_patch_" + patch_hash[:12] + ".json"),
    {
        "patch_sha256": patch_hash,
        "scope": "Only CSV publication retry; model/data/config unchanged",
        "source": patch_file.name,
    },
)

resumed = reporting.launch(
    ROOT,
    "original_checks",
    CONFIG,
    [worker_setup, worker_patch, worker_run],
)
if resumed != RUN_DIR:
    raise RuntimeError("Unexpected run directory; inspect result before continuing.")
print("ORIGINAL_CHECKS_COMPLETE", resumed)
print("Proof:", resumed.with_name(resumed.name + "_review.zip"))


RUN_DIR D:\UIT\DS317\BAF_project\results\advisor_runs\original_checks_582e4a8ada1a
BEGIN_STAGE 1
RESUME_ORIGINAL_RUN D:\UIT\DS317\BAF_project\results\advisor_runs\original_checks_582e4a8ada1a
END_STAGE 1
BEGIN_STAGE 2
CSV_RETRY_PATCH_ACTIVE: toi da 5 giay, giu cach ghi atomic
END_STAGE 2
BEGIN_STAGE 3
                             check  max_abs_error  pass
                 CB_weights_beta_0   0.000000e+00  True
               CB_weights_beta_0.9   0.000000e+00  True
            CB_weights_beta_0.9999   1.332268e-15  True
          CB_weights_beta_0.999999   1.201261e-13  True
           CB_CE_loss_and_gradient   0.000000e+00  True
          CB_focal_sigmoid_formula   0.000000e+00  True
LRE_independent_gradient_alignment   2.775558e-17  True
    MWN_author_MetaModule_one_step   0.000000e+00  True
NUMERICAL_CHECKS_PASS; this does not reproduce full-paper accuracy
ORIGINAL_FIT_REUSED mnist ce 42
ORIGINAL_FIT_REUSED mnist lre 42
ORIGINAL_FIT_REUSED mnist ce 317
ORIGINAL_FIT_REUSED mnist lr